# 2.4 Python List-Array Datatype

**Prerequisites:** 2.3 Python Tuple Datatype
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Creating lists, and converting other types into them
- Indexing, slicing, slice assignment and slice deletion
- **Adding & removing** — `append()`/`extend()`/`insert()` vs `remove()`/`del`/`pop()`, plus three classic traps
- **Searching, counting & aggregating** — `in`, `index()`, `count()`, `min()`/`max()`/`sum()`/`any()`/`all()`
- **Sorting & reordering** — `sorted()` vs `.sort()`, `key=` functions, sort stability, `reverse()`
- **Copying** — alias vs shallow copy vs deep copy
- **Nesting** — lists of lists as grids and matrices
- When `array` or NumPy beats a plain list

The examples run on the kinds of lists real programs keep: a build server's task queue, a day of sensor readings, a deployment checklist, a status grid.

---

## List in python:
- List in python are ordered collection of items of different data types.  
- The items within the list are separated by commas (,) and delimited by square brackets `[]`.
- List support forward and backward indexing.
- List are mutable.
- List are iterable objects.

### Create empty list:

In [ ]:
list1 = []
list2 = list()
print(list1, type(list1), len(list1))
print(list2, type(list2), len(list2))

### Create non-empty list:

In [ ]:
# A release record: build number, service, version, passed QA?, (owners)
release = [217, 'web-api', 'v2.4.1', True, ('aditya', 'priya')]
print(release, type(release), len(release))

In [ ]:
# Build a list element by element (deterministic; no input())
values = ['101', 'Aditya', '95.6']   # e.g. fields parsed from a CSV row
l = []
for v in values:
    l.append(v)
print(l)

# ⚠️ The original used eval(input()) in a loop — never eval() user input (see 1.3's
# warning); convert explicitly with int()/float(), or use ast.literal_eval for literals.

### Convert Different datatypes into list:

In [ ]:
chars = list('deploy')        # a string explodes into its characters
print(chars, type(chars))

In [ ]:
readings = list((21.5, 22.0, 21.8))    # tuple -> list (now mutable)
print(readings, type(readings))

## Accessing elements of list:
- In Python, we use brackets `[]` after an object to call its index.
- **Forward indexing** starts with 0.
- **Backward indexing** starts with -1.

### Scenario: a day of sensor readings
A monitoring agent samples the server-room temperature once an hour and appends each value to a list. Indexing answers *"what was reading N?"*; slicing answers *"the first three"*, *"the three most recent"*, *"every other one"*.

### List Indexing:

In [ ]:
# Hourly server-room temperatures, oldest first
temps = [21.5, 22.0, 21.8, 23.1, 22.6, 24.0, 23.4]

print(temps[0])          # first reading of the day
print(temps[-1])         # most recent reading
print(temps[3])

# Indexing reaches into nested items too
print(release[4][0])     # first owner inside the tuple
print(release[1][0])     # first character of the service name

### List Index Slicing:

In [ ]:
print(temps[:3])         # the first three readings
print(temps[-3:])        # the three most recent
print(temps[1:5])
print(temps[::2])        # every other reading
print(temps[::-1])       # newest first — a reversed COPY
print(temps)             # original untouched by any slice above

In [ ]:
# Lists are MUTABLE: assign straight into an index
temps[0] = 21.9          # correct a bad first reading
print(temps)

### Slice assignment and deletion

Slicing a list gives you a **new list**. But a slice can also appear on the **left** of an
assignment, where it replaces that region in place — and the replacement need not be the
same length.

```
lst[start : stop : step]
      |      |      |
      |      |      +-- stride; negative walks backwards
      |      +--------- exclusive
      +----------------- inclusive; defaults to 0
```

Note the difference between `a[:] = [...]` (mutates the existing object, visible to every
alias) and `a = [...]` (rebinds the name, invisible to aliases). This distinction is the
whole subject of **2.7**.

In [ ]:
nums = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Slicing READS a new list - the original is untouched
print("nums[2:5]   :", nums[2:5])
print("nums[::2]   :", nums[::2])
print("nums[::-1]  :", nums[::-1], "  <- reversed copy")
print("original    :", nums)

# Slice ASSIGNMENT writes back - and the lengths need not match
nums[2:5] = ["a", "b"]           # replace 3 items with 2
print("\nafter nums[2:5] = ['a','b']:", nums)

nums[2:4] = [2, 3, 4]            # replace 2 items with 3
print("after nums[2:4] = [2,3,4] :", nums)

# Extended slices DO require matching lengths
try:
    nums[::2] = [0]
except ValueError as exc:
    print("\nExtended slice:", exc)

# Deleting a slice
data = list(range(10))
del data[::2]
print("\ndel data[::2]:", data)

# The whole-list slice: replace CONTENTS in place, keeping the same object
a = [1, 2, 3]
b = a                            # b is an alias
a[:] = [7, 8, 9]                 # mutates the shared object
print("\na[:] = [7,8,9] -> a:", a, "| b:", b, "  <- b sees it too")

a = [1, 2, 3]
b = a
a = [7, 8, 9]                    # REBINDS a; b still points at the old object
print("a = [7,8,9]    -> a:", a, "| b:", b, "  <- b does not")

## Adding and removing elements
A list earns its keep when it changes size. Picture the **task queue** of a small build server:

- a new job arrives → `append()` it at the back
- several jobs arrive at once → `extend()`
- an urgent hotfix must jump the queue → `insert(0, ...)`
- a job is cancelled by name → `remove()`
- the worker takes the next job → `pop()`

All of these change the list **in place** — no new list is created.

- **append():** Adds its argument as a **single object** at the end of a list.
    - **Syntax:** `object.append(element)`
    - **NOTE: append() takes exactly one argument.**

In [ ]:
queue = ['build-217', 'test-217']
queue.append('deploy-217')                    # one job joins the back
print(queue)

queue.append(['notify-217', 'cleanup-217'])   # ⚠️ the LIST arrives as ONE element
print(queue)
del queue[-1]                                 # undo that (del is covered below)

- **extend():** Iterate over its argument(s) and **add each object** at the end of  list.
    - **Syntax:** `object.extend(sequence)`
    - **NOTE: extend() takes exactly one argument.**

In [ ]:
print(queue)
queue.extend(['notify-217', 'cleanup-217'])   # each element joins individually
print(queue)

queue.extend('QA')      # ⚠️ a string is iterable — this adds 'Q' and 'A'
print(queue)
del queue[-2:]          # undo

- **insert():** It inserts the object at a **given index** in list.
    - **Syntax:** `object.insert(index, element)`

In [ ]:
print(queue)
queue.insert(0, 'hotfix-218')      # urgent job jumps to the front
print(queue)

- **remove():** Removes the first occurence of element from list. Raise ValueError if element not in list. 
    - **Syntax:** `list.remove(element)`
    - **NOTE: remove() takes exactly one argument.**

In [ ]:
print(queue)
queue.remove('test-217')           # cancel a job BY VALUE (first match)
print(queue)
# queue.remove('missing-job')      # ValueError: list.remove(x): x not in list

- **pop():** It removes the object at **specified index** or **by default pop last object**. Also return the popped object.
    - **Syntax:** `list.pop([i])` — pops the element at index `i`, or the last element if `i` is omitted. The index is positional only; `pop()` accepts no keywords.
    - **NOTE: pop() takes at most 1 argument**

In [ ]:
print(queue)
next_job = queue.pop(0)            # take the next job from the FRONT...
print('running :', next_job)
last_job = queue.pop()             # ...no index means the LAST element
print('deferred:', last_job)
print(queue)
# For a heavy-duty queue use collections.deque — pop(0) on a long list is
# slow because everything shifts left (see Best Practices).

- **clear():** It removes all the elements from the list.
    - **Syntax**: `list.clear()`

In [ ]:
done = queue.copy()      # keep the original; .copy() is explained below
done.clear()             # empty the list in place
print('done  :', done)
print('queue :', queue)

- **del:** A *statement* (not a function — no parentheses) that deletes a single element, a slice, or the whole list.
    - **Syntax:** `del list[i]`, `del list[start:stop:step]`, `del list`

In [ ]:
jobs = ['a', 'b', 'c', 'd', 'e', 'f']
del jobs[1]              # delete AT an index
print(jobs)
del jobs[1:3]            # delete a slice
print(jobs)
del jobs                 # delete the whole name
# print(jobs)            # NameError: name 'jobs' is not defined

- **Concatenation Operator(+):**
    - Process to combine two list.

In [ ]:
morning = ['backup-db', 'rotate-logs']
evening = ['report-usage', 'archive-metrics']
day_plan = morning + evening       # a NEW list; the originals are untouched
print(day_plan)
print(morning)

- **Replication Operator(*):**
    - Process of repeating elements of list multiple times.

In [ ]:
board = ['pending'] * 4            # initialise 4 slots to the same value
print(board)
# ⚠️ fine for strings/numbers — but read Trap 2 below before doing this with LISTS

---

### ⚠️ Three list traps worth knowing before you need them

In [ ]:
# ---- Trap 1: modifying a list while iterating over it ----
numbers = [1, 2, 2, 3, 4, 4, 5]

buggy = numbers.copy()
for n in buggy:
    if n % 2 == 0:
        buggy.remove(n)          # shifts everything left, loop index marches on
print("Buggy  :", buggy, "  <- a 2 and a 4 survived; the loop skipped them")

# Fix A: iterate over a copy, mutate the original
ok_a = numbers.copy()
for n in ok_a[:]:                # the slice is a snapshot
    if n % 2 == 0:
        ok_a.remove(n)
print("Fix A  :", ok_a)

# Fix B (preferred): build a new list
ok_b = [n for n in numbers if n % 2 != 0]
print("Fix B  :", ok_b)


# ---- Trap 2: * copies references, not objects ----
grid_bad = [[0] * 3] * 3         # THREE references to ONE row
grid_bad[0][0] = 1
print("\nBad grid :", grid_bad, "  <- all rows changed")

grid_ok = [[0] * 3 for _ in range(3)]     # a fresh row each iteration
grid_ok[0][0] = 1
print("Good grid:", grid_ok)

print("\nSame object?", grid_bad[0] is grid_bad[1], "vs", grid_ok[0] is grid_ok[1])


# ---- Trap 3: remove() vs del vs pop() ----
items = ["a", "b", "c", "b"]
print("\nstart      :", items)

items.remove("b")                # first element EQUAL TO "b"
print("remove('b'):", items)

del items[0]                     # element AT index 0
print("del [0]    :", items)

popped = items.pop()             # remove and RETURN the last
print("pop()      :", items, "| returned:", popped)

## Searching, counting & aggregating
Scenario: a **deployment checklist**. Before you flip the switch you ask: is `'smoke-test'` on the list? At which position? How many times does `'restart-api'` appear? The membership operator and two list methods answer those; the built-ins `min()`/`max()`/`sum()`/`any()`/`all()` then squeeze a whole list of readings down to one answer.

- **Membership Operator(in/not in):** 

In [ ]:
checklist = ['migrate-db', 'update-config', 'restart-api', 'smoke-test', 'restart-api']

print('smoke-test' in checklist)
print('rollback' in checklist)
print('rollback' not in checklist)

- **index():** It returns the lowest index of occurrence of element in list. And raises ValueError if element is not found.
    - **Syntax:** `object.index(element, start, stop)`

In [ ]:
print(checklist.index('restart-api'))       # index of the FIRST match
print(checklist.index('restart-api', 3))    # search again, starting at index 3
# print(checklist.index('rollback'))        # ValueError when absent — test with `in` first

- **count():** Return the number of times an element appears in the list.
    - **Syntax:** `object.count(element)`

In [ ]:
print(checklist.count('restart-api'))
print(checklist.count('rollback'))          # 0 — counting never raises

### Aggregating: `min()`, `max()`, `sum()`, `any()`, `all()`
These are **built-in functions**, not list methods — they accept any iterable and reduce it to a single value. Together with `len()` they answer most *"summarise this list"* questions in one line.

In [ ]:
temps = [21.5, 22.0, 21.8, 23.1, 22.6, 24.0, 23.4]

print('coldest :', min(temps))
print('warmest :', max(temps))
print('average :', round(sum(temps) / len(temps), 2))

In [ ]:
alerts = [0, 0, 1, 0]              # 1 = that hour's sensor raised an alert
print(any(alerts))                 # True — at least one truthy value

checks = [True, True, True]
print(all(checks))                 # True — every health check passed
print(all([True, False, True]))

## Sorting and reordering
Request latencies arrive in arrival order; a report wants them ranked. Python gives two routes — `sorted()` builds a **new** list, `.sort()` reorders **in place** — plus `reverse()` and reversed slices for flipping order. The differences matter enough that they get a full comparison further down.

In [ ]:
latencies = [182, 45, 91, 230, 12, 91]     # request latencies in ms

print(sorted(latencies))                   # ascending, in a NEW list
print(sorted(latencies, reverse=True))     # descending
print(latencies)                           # arrival order preserved

In [ ]:
# Strings sort by Unicode code points — all uppercase before all lowercase
services = ['auth', 'Billing', 'cache', 'API']
print(sorted(services))
print(sorted(services, key=str.casefold))   # case-insensitive ordering

- **sort():** Sort the items of the list in place.
    - **Syntax:** `list.sort(key=None, reverse=False)`

In [ ]:
lat = latencies.copy()
lat.sort()                     # in place, ascending
print(lat)
lat.sort(reverse=True)         # in place, descending
print(lat)

In [ ]:
logs = ['app.log', 'frontend.log', 'db.log', 'api.log']
logs.sort(key=len)             # compare by the RESULT of len() — shortest first
print(logs)

- **reverse():** Reverse the elements of the list in place.

In [ ]:
history = ['deploy-215', 'deploy-216', 'deploy-217']   # oldest first
history.reverse()                # in place — newest first now
print(history)
print(history[::-1])             # a reversed COPY instead...
print(history)                   # ...leaving the list itself alone

---

### `sorted()` vs `.sort()` — and sorting properly

This distinction trips up almost everyone once, and the failure is silent.

| | `sorted(iterable)` | `list.sort()` |
|---|---|---|
| **Returns** | A **new list** | **`None`** |
| **Original** | Unchanged | Mutated in place |
| **Works on** | Any iterable | Lists only |
| **Use when** | You need both orders, or the input isn't a list | You want to reorder and don't need the original |

Both accept the same two keyword arguments:

- **`key=`** — a function called on each element; the sort compares the *results*.
- **`reverse=`** — `True` for descending.

> **Python's sort is stable.** Elements comparing equal keep their original relative order.
> This is a documented guarantee, and it's what makes multi-pass sorting work: sort by the
> least-significant key first, then by the most-significant.

In [ ]:
numbers = [5, 2, 9, 1, 7]

# sorted() -> returns a NEW list, original untouched
new = sorted(numbers)
print("sorted() returned:", new)
print("original         :", numbers)

# .sort() -> sorts IN PLACE, returns None
result = numbers.sort()
print("\n.sort() returned :", result, "  <- None, not the list!")
print("original         :", numbers, "  <- mutated")

# THE CLASSIC BUG:
data = [3, 1, 2]
data = data.sort()          # looks reasonable, destroys your data
print("\ndata = data.sort() ->", data)

# sorted() works on ANY iterable; .sort() only exists on lists
print("\nsorted on a tuple:", sorted((3, 1, 2)))
print("sorted on a str  :", sorted("python"))
print("sorted on a dict :", sorted({"b": 1, "a": 2}))   # sorts the KEYS

In [ ]:
from operator import itemgetter

students = [
    ("Aditya", 78, "CBSE"),
    ("Priya", 92, "ICSE"),
    ("Rahul", 78, "CBSE"),
    ("Sneha", 85, "CBSE"),
]

# Sort by marks descending, using a lambda
print("By marks desc:")
for s in sorted(students, key=lambda row: row[1], reverse=True):
    print("  ", s)

# itemgetter is faster and reads better for plain index/key access
print("\nSame, with itemgetter:")
print("  ", sorted(students, key=itemgetter(1), reverse=True)[0])

# Multi-key sort: marks descending, then name ascending as a tie-break.
# Because the sort is STABLE, you can also do it in two passes, least-significant first.
print("\nMarks desc, then name asc:")
for s in sorted(students, key=lambda row: (-row[1], row[0])):
    print("  ", s)

# Stability demonstrated: equal marks keep their original relative order
print("\nStable sort keeps Aditya before Rahul (both 78):")
for s in sorted(students, key=itemgetter(1)):
    print("  ", s)

# Case-insensitive string sorting
words = ["banana", "Apple", "cherry", "apple"]
print("\nDefault (code points):", sorted(words))
print("Case-insensitive:", sorted(words, key=str.casefold))

### Exercise: check whether two strings are anagrams:
- 'Python is great' 
- 'Thorn pestiagy'
- These two strings are anagram of each other

In [ ]:
x = 'Python is great'
y = 'Thorn pestiagy'
# Interactive variant:
# x = input('Enter string 1: ')
# y = input('Enter string 2: ')

x_clean = x.lower().replace(' ', '')
y_clean = y.lower().replace(' ', '')

print(list(x_clean))   # list() explodes a string into its characters

# sorted() accepts a string directly and returns a sorted list of characters,
# so two strings are anagrams iff their sorted character lists are equal
if sorted(x_clean) == sorted(y_clean):
    print(f'{x!r} and {y!r} are anagrams')
else:
    print(f'{x!r} and {y!r} are NOT anagrams')

## Copying a list
`b = a` does **not** copy a list — it gives the same list a second name. This is the single most common list surprise, and it is why `.copy()` exists.

### Python List methods
- **copy():** Returns a **shallow copy** of the list. Three ideas that are easy to conflate:
    - **Alias (`b = a`):** no copy at all — both names refer to the *same* list, so a change made through either name is visible through the other (demo below).
    - **Shallow copy (`a.copy()`):** a new outer list whose elements are the *same* objects. Replacing an element of the copy leaves the original alone, but mutating a shared *nested* object shows up in both.
    - **Deep copy (`copy.deepcopy(a)`):** recursively copies nested objects too — nothing is shared.
- The note after the demos below covers the shallow/deep distinction; the full treatment is in **2.7 Mutability, Copying, Nesting & Unpacking**.

In [ ]:
print(checklist)
temp = checklist                 # an ALIAS — the same list under a second name
temp[0] = 'migrate-db-v2'
print(temp)
print(checklist)                 # the "original" changed too — same object

In [ ]:
print(checklist)
snapshot = checklist.copy()      # a real (shallow) copy — new outer list
snapshot[0] = 'migrate-db'
print(snapshot)
print(checklist)                 # original unchanged this time

> **`.copy()` is shallow.** It builds a new outer list, but the *elements* are the same
> objects. For a flat list of numbers or strings that is indistinguishable from a real copy;
> for a list of lists it very much is not:
>
> ```python
> original = [[1, 2], [3, 4]]
> shallow = original.copy()
> shallow[0].append(99)        # also visible through `original`
> ```
>
> `copy.deepcopy()` is the fix. Copying, mutability and aliasing get a notebook of their
> own — **2.7 Mutability, Copying, Nesting & Unpacking**.

## Nesting: lists of lists
A list element can itself be a list — that gives you rows and columns: a rack of servers, a game board, a matrix of numbers. Two indexes deep reads `outer[row][column]`.

(Trap 2 above — `[[0]*3]*3` — is exactly about building these safely.)

In [ ]:
# A 2-D list: [row][column] — three racks, two servers each
racks = [['web-01', 'web-02'], ['db-01', 'db-02'], ['cache-01', 'cache-02']]

print(racks[1])          # the second row
print(racks[1][0])       # first server in that row
print(len(racks), 'rows x', len(racks[0]), 'columns')

In [ ]:
# Build a 3x4 grid of zeros THE SAFE WAY (a fresh row each time — see Trap 2)
grid = [[0] * 4 for _ in range(3)]
grid[0][0] = 1
print(grid[0])
print(grid[1])           # unaffected — the rows are separate lists

### Parsing lists from text
Data files and network payloads arrive as strings; `split()` plus a conversion turns one line into a list, and repeating that per line builds a matrix.

In [ ]:
# 1-D array parsed from a comma-separated string (e.g. one CSV line)
csv_line = '2,11,4,5,3,5,8,11'
readings = list(map(int, csv_line.split(',')))
print(readings)
# Interactive variant:
# readings = list(map(int, input('Enter comma-separated elements: ').split(',')))

In [ ]:
# 2-D array built row by row from comma-separated strings
csv_rows = ['1,2,3', '4,5,6']
matrix = []
for row in csv_rows:
    matrix.append(list(map(int, row.split(','))))
print(matrix)
# Interactive variant:
# n = int(input('Enter no. of rows: '))
# matrix = [list(map(int, input('Enter comma-separated row: ').split(','))) for _ in range(n)]

---

## `array` module: when a list is the wrong tool

- `array` is not a built-in type like `str` or `int` — you must `import array` to use it.
- Unlike a list, an `array` stores **raw values of one fixed numeric type**, packed
  contiguously in memory. No per-element object overhead.
- **Syntax:** `array.array(typecode, [initializers])`

### Type codes

> **Corrected from the original note:** the original table listed `'c' - character of size
> 1 byte`. That typecode was **removed in Python 3** — it only existed in Python 2. The
> table below is the current one.
>
> Two more version notes: `'u'` is **deprecated** (removal is scheduled, so avoid it in new code), and **Python 3.13 added `'w'`** (`Py_UCS4`, 4 bytes) as its replacement.

| Code | C type | Minimum bytes |
|---|---|---|
| `'b'` / `'B'` | signed / unsigned char | 1 |
| `'u'` | Unicode character (`wchar_t`) | 2 |
| `'w'` | Unicode character (`Py_UCS4`) | 4 |
| `'h'` / `'H'` | signed / unsigned short | 2 |
| `'i'` / `'I'` | signed / unsigned int | 2 |
| `'l'` / `'L'` | signed / unsigned long | 4 |
| `'q'` / `'Q'` | signed / unsigned long long | 8 |
| `'f'` | float | 4 |
| `'d'` | double | 8 |

### So when would you actually use it?

Almost never, honestly. The decision tree in practice:

| Situation | Use |
|---|---|
| General-purpose collection, mixed types | **`list`** |
| Millions of numbers, and you need the memory back | **`array`** |
| Numerical work: maths on whole arrays, matrices, slicing by axis | **NumPy** `ndarray` |
| Reading/writing a binary file format with fixed-width fields | **`array`** or `struct` |

`array` occupies a narrow middle ground: more compact than a list, far less capable than
NumPy. Know it exists; reach for NumPy when you're doing real numeric work.

In [ ]:
import array

In [ ]:
ar1 = array.array('i',[1,2,3,4,5])
print(ar1)
print(type(ar1))

In [ ]:
print(ar1[1])

In [ ]:
print(ar1[2:5])

In [ ]:
ar1.append(6)
print(ar1)

In [ ]:
ar1.extend([9,3])
print(ar1)

In [ ]:
ar1.insert(3,1)
print(ar1)

In [ ]:
ar1.remove(3)
print(ar1)

---

## Common Mistakes & Pitfalls

1. **Modifying a list while iterating over it.** Removing items shifts the indices under the loop and silently skips elements. Iterate over a copy, or build a new list.
2. **`[[0] * 3] * 3` for a grid.** The outer `*` copies the *reference*, so all three rows are the same list. Use a comprehension.
3. **Confusing `append()` with `extend()`.** `append("END")` adds one string; `extend("END")` adds three characters.
4. **Expecting `sort()` to return the sorted list.** It sorts in place and returns `None`. `x = my_list.sort()` sets `x` to `None` — use `sorted()` if you want a value back.
5. **Assuming `list.copy()` is a deep copy.** It copies the outer list only; nested objects are still shared. See **2.7**.
6. **Using `remove()` when you mean `del` or `pop()`.** `remove(2)` removes the first element *equal to* 2; `del lst[2]` removes the element *at index* 2.
7. **Using `index()` without catching `ValueError`.** It raises when the item is absent — check with `in` first, or catch it.
8. **Using a list for membership tests on large data.** `x in my_list` is O(n). A `set` is O(1).

## Best Practices

- Use a **list** for ordered, variable-length, homogeneous data; a **tuple** for fixed records.
- Build lists with comprehensions where it stays readable (see **03 Flow Control**).
- Use `sorted()` when you need a new list, `.sort()` when you want to reorder in place.
- Pass `key=` to sort by a computed value; never decorate-sort-undecorate by hand.
- Use `enumerate()` instead of `range(len(x))` when you need both index and value.
- Use `collections.deque` when you push/pop at the front — `list.insert(0, x)` is O(n).
- Reach for a `set` the moment membership testing shows up in a loop.

## Practice Exercises

Try these before moving on.

1. Build a 3x3 grid of zeros. Set `grid[0][0] = 1` and confirm the other rows are unaffected.
2. Remove every even number from `[1,2,3,4,5,6]` — first the buggy way (mutating while iterating), then correctly. Explain the difference in output.
3. Sort a list of `(name, marks)` tuples by marks descending, then by name ascending as a tie-break.
4. Flatten `[[1,2],[3,4],[5,6]]` into `[1,2,3,4,5,6]` two different ways.
5. Given a list with duplicates, produce a duplicate-free list that **preserves original order**.
6. Time `x in big_list` against `x in big_set` for 100,000 elements.
7. Rotate a list left by `k` positions using slicing only.
8. Explain why `lst.sort()` returns `None` and why that is a deliberate design choice.